In [ ]:
import pandas as pd
import numpy as np
import re
import os
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam


In [ ]:
from google.colab import files
uploaded = files.upload()

df = pd.read_csv(next(iter(uploaded)))
df.head()


Saving train.csv to train.csv


,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [ ]:
def clean_text(text):
    text = re.sub(r"http\S+", "", text)          # Remove URLs
    text = re.sub(r"@\w+", "", text)             # Remove mentions
    text = re.sub(r"#", "", text)                # Remove #
    text = re.sub(r"[^A-Za-z0-9 ]+", "", text)   # Remove special chars
    text = text.lower()
    return text

df["clean_text"] = df["text"].astype(str).apply(clean_text)


In [ ]:
MAX_WORDS = 20000
MAX_LEN = 40

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(df["clean_text"])

sequences = tokenizer.texts_to_sequences(df["clean_text"])
X = pad_sequences(sequences, maxlen=MAX_LEN, padding='post')

y = df["target"].values


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42)


In [ ]:
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip


--2025-11-23 10:19:05--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2025-11-23 10:19:05--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2025-11-23 10:19:06--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

In [ ]:
embeddings_index = {}
with open("glove.6B.100d.txt", encoding="utf-8") as f:
    for line in tqdm(f):
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype="float32")
        embeddings_index[word] = vector


400000it [00:14, 27354.12it/s]


In [ ]:
embedding_dim = 100
word_index = tokenizer.word_index
num_words = min(MAX_WORDS, len(word_index) + 1)

embedding_matrix = np.zeros((num_words, embedding_dim))

for word, i in word_index.items():
    if i < MAX_WORDS:
        embedding_vector = embeddings_index.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector


In [ ]:
import keras
from keras.layers import (
    Embedding, Bidirectional, LSTM, Conv1D, GlobalMaxPooling1D,
    Dense, Dropout, Input
)
from keras.models import Model
import keras.ops as ops   # <-- IMPORTANT: use keras.ops instead of tf


embedding_layer = Embedding(
    num_words,
    embedding_dim,
    weights=[embedding_matrix],
    input_length=MAX_LEN,
    trainable=True
)

input_ = Input(shape=(MAX_LEN,))

x = embedding_layer(input_)

# -----------------------
# BiLSTM Layer
# -----------------------
lstm_out = Bidirectional(
    LSTM(128, return_sequences=True, dropout=0.3)
)(x)

# -----------------------
# Attention Layer (Keras-native)
# -----------------------
score = Dense(1, activation='tanh')(lstm_out)      # shape: (batch, seq, 1)
attention_weights = ops.softmax(score, axis=1)     # KERAS OPS (NO ERROR)
context_vector = ops.sum(attention_weights * lstm_out, axis=1)

# -----------------------
# CNN Branch
# -----------------------
cnn = Conv1D(filters=128, kernel_size=3, activation='relu')(lstm_out)
cnn = GlobalMaxPooling1D()(cnn)

# -----------------------
# Merge LSTM-Attention + CNN Features
# -----------------------
combined = ops.concatenate([context_vector, cnn], axis=1)

# -----------------------
# Dense Layers
# -----------------------
dense = Dense(128, activation='relu')(combined)
dense = Dropout(0.4)(dense)
dense = Dense(64, activation='relu')(dense)
dense = Dropout(0.3)(dense)

# Output layer
output_ = Dense(1, activation='sigmoid')(dense)

model = Model(input_, output_)

model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 40)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 40, 100)   │  1,578,000 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_2     │ (None, 40, 256)   │    234,496 │ embedding_2[0][0] │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 40, 1)     │        257 │ bidirectional_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ softmax (Softmax)   │ (None, 40, 1)     │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 40, 256)   │          0 │ softmax[0][0],    │
│                     │                   │            │ bidirectional_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 38, 128)   │     98,432 │ bidirectional_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sum (Sum)           │ (None, 256)       │          0 │ multiply[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 128)       │          0 │ conv1d[0][0]      │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 384)       │          0 │ sum[0][0],        │
│ (Concatenate)       │                   │            │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 128)       │     49,280 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 128)       │          0 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 64)        │      8,256 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 64)        │          0 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 1)         │         65 │ dropout_3[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,968,786 (7.51 MB)

 Trainable params: 1,968,786 (7.51 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=12,
    batch_size=64,
    verbose=1
)


Epoch 1/12
96/96 ━━━━━━━━━━━━━━━━━━━━ 33s 275ms/step - accuracy: 0.5979 - loss: 0.6693 - val_accuracy: 0.7814 - val_loss: 0.5554
Epoch 2/12
96/96 ━━━━━━━━━━━━━━━━━━━━ 25s 262ms/step - accuracy: 0.7533 - loss: 0.5567 - val_accuracy: 0.8030 - val_loss: 0.4652
Epoch 3/12
96/96 ━━━━━━━━━━━━━━━━━━━━ 25s 263ms/step - accuracy: 0.7807 - loss: 0.5027 - val_accuracy: 0.8116 - val_loss: 0.4375
Epoch 4/12
96/96 ━━━━━━━━━━━━━━━━━━━━ 28s 290ms/step - accuracy: 0.7963 - loss: 0.4609 - val_accuracy: 0.8201 - val_loss: 0.4258
Epoch 5/12
96/96 ━━━━━━━━━━━━━━━━━━━━ 27s 282ms/step - accuracy: 0.8011 - loss: 0.4489 - val_accuracy: 0.8148 - val_loss: 0.4283
Epoch 6/12
96/96 ━━━━━━━━━━━━━━━━━━━━ 27s 285ms/step - accuracy: 0.8075 - loss: 0.4401 - val_accuracy: 0.8234 - val_loss: 0.4153
Epoch 7/12
96/96 ━━━━━━━━━━━━━━━━━━━━ 24s 253ms/step - accuracy: 0.8184 - loss: 0.4225 - val_accuracy: 0.8260 - val_loss: 0.4140
Epoch 8/12
96/96 ━━━━━━━━━━━━━━━━━━━━ 42s 259ms/step - accuracy: 0.8174 - loss: 0.4185 - val_accu

In [ ]:
loss, acc = model.evaluate(X_val, y_val)
print(f"Validation Accuracy: {acc*100:.2f}%")
print(f"Validation Loss: {loss:.4f}")


48/48 ━━━━━━━━━━━━━━━━━━━━ 3s 73ms/step - accuracy: 0.8134 - loss: 0.4335
Validation Accuracy: 82.21%
Validation Loss: 0.4175


In [ ]:
sample = ["Huge flooding happening downtown, houses destroyed!"]
sample_clean = [clean_text(s) for s in sample]
sample_seq = tokenizer.texts_to_sequences(sample_clean)
sample_pad = pad_sequences(sample_seq, maxlen=MAX_LEN)

pred = model.predict(sample_pad)[0][0]
print("Prediction:", ("Disaster Tweet" if pred > 0.5 else "Non-Disaster Tweet"), "| Score:", pred)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 423ms/step
Prediction: Disaster Tweet | Score: 0.97856647


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
model.save("/content/drive/MyDrive/disaster_tweet_model.keras")


In [ ]:
model.save_weights("/content/drive/MyDrive/disaster_tweet_model.weights.h5")


In [ ]:
import pickle

with open("tokenizer_disaster.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("Tokenizer saved!")


Tokenizer saved!
